# Traceprop — Multi-Source Credit Risk Provenance Case Study

**Purpose:** VLDB 2027 Section 6.6 — Real-world multi-table ETL provenance case study.

**What this notebook demonstrates:**
1. Traceprop tracks row-level lineage across **3 source tables** (application, bureau, previous_application)
2. After joins and aggregations, `trace_to_file()` identifies which source rows influenced a prediction
3. ETL overhead vs plain pandas
4. Query latency on the joined lineage DAG

**Dataset:** Synthetic credit risk dataset mimicking the Home Credit 3-table schema.
Fully reproducible — no credentials or downloads required.

**Schema:**
- `application` — 1 row per loan applicant (income, employment, loan amount)
- `bureau` — credit bureau history (0–10 rows per applicant)
- `previous_application` — prior loan applications (0–5 rows per applicant)

**Checkpoints saved to:** `Google Drive/Traceprop/homecredit/`

**Results JSON:** `results/exp5_homecredit_provenance.json`

---
**Run order:** Cells 1 → 12 sequentially. Checkpoints allow resuming from any stage.

**Hardware:** CPU only (no GPU needed)

In [ ]:
# ============================================================
# Cell 1 — Mount Google Drive + define paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT   = '/content/drive/MyDrive/Traceprop/homecredit'
DATA_DIR     = os.path.join(DRIVE_ROOT, 'data')
CKPT_DIR     = os.path.join(DRIVE_ROOT, 'checkpoints')
RESULTS_DIR  = os.path.join(DRIVE_ROOT, 'results')

for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Directories ready:')
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    print(' ', d)

In [ ]:
# ============================================================
# Cell 2 — Install dependencies
# ============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip_install('scikit-learn')
pip_install('scipy')

print('Dependencies installed.')

In [ ]:
# ============================================================
# Cell 3 — Generate synthetic multi-table credit risk dataset
#
# Mimics Home Credit Default Risk schema (3 tables, realistic
# distributions). No credentials or downloads required.
#
# Schema:
#   application       — 1 row per applicant  (N_APPS rows)
#   bureau            — 0–10 rows/applicant  (credit history)
#   previous_application — 0–5 rows/applicant (prior loans)
#
# Checkpoint: saves raw CSVs to Drive so later cells can reload.
# ============================================================
import os, pickle
import numpy as np
import pandas as pd

CKPT_DATA = os.path.join(CKPT_DIR, 'data_generated.flag')

N_APPS = 20_000   # number of loan applicants
RNG    = np.random.default_rng(42)

if os.path.exists(CKPT_DATA):
    print('Data checkpoint found — loading from Drive...')
    app_raw  = pd.read_csv(os.path.join(DATA_DIR, 'application.csv'))
    bur_raw  = pd.read_csv(os.path.join(DATA_DIR, 'bureau.csv'))
    prev_raw = pd.read_csv(os.path.join(DATA_DIR, 'previous_application.csv'))
    print(f'Loaded: app={len(app_raw):,}  bureau={len(bur_raw):,}  prev={len(prev_raw):,}')

else:
    print(f'Generating synthetic credit risk dataset (N={N_APPS:,} applicants)...')

    # ---- application table (1 row per applicant) ----
    sk_ids = np.arange(100_000, 100_000 + N_APPS)

    income     = RNG.lognormal(mean=11.0, sigma=0.5, size=N_APPS).clip(20_000, 1_000_000)
    credit     = income * RNG.uniform(2, 8, size=N_APPS)
    annuity    = credit / RNG.uniform(24, 96, size=N_APPS)
    goods      = credit * RNG.uniform(0.7, 0.95, size=N_APPS)
    days_birth = -RNG.integers(18*365, 65*365, size=N_APPS)
    days_emp   = -RNG.integers(0, 20*365, size=N_APPS)
    cnt_fam    = RNG.integers(1, 6, size=N_APPS).astype(float)
    gender     = RNG.choice(['M', 'F'], size=N_APPS)
    edu        = RNG.choice(['Secondary', 'Higher', 'Incomplete higher'], size=N_APPS)

    # Target: ~8% default rate, correlated with income/credit ratio
    log_odds = (
        -3.5
        + 0.5  * (credit / income - 4) / 2          # high credit-to-income → riskier
        - 0.3  * (days_emp / -365)                   # longer employment → safer
        + 0.2  * (cnt_fam - 3)                       # more dependents → riskier
        + RNG.normal(0, 0.5, size=N_APPS)
    )
    target = (RNG.uniform(size=N_APPS) < 1 / (1 + np.exp(-log_odds))).astype(int)

    app_raw = pd.DataFrame({
        'SK_ID_CURR':       sk_ids,
        'AMT_INCOME_TOTAL': income.round(2),
        'AMT_CREDIT':       credit.round(2),
        'AMT_ANNUITY':      annuity.round(2),
        'AMT_GOODS_PRICE':  goods.round(2),
        'DAYS_BIRTH':       days_birth,
        'DAYS_EMPLOYED':    days_emp,
        'CNT_FAM_MEMBERS':  cnt_fam,
        'CODE_GENDER':      gender,
        'NAME_EDUCATION_TYPE': edu,
        'TARGET':           target,
    })

    # ---- bureau table (0–10 rows per applicant) ----
    bureau_rows = []
    bur_id = 200_000
    for sk_id, tgt in zip(sk_ids, target):
        n = int(RNG.integers(0, 11))   # 0–10 credit bureau entries
        for _ in range(n):
            debt        = RNG.lognormal(9, 1) * (1 + 0.5 * tgt)
            overdue     = max(0.0, RNG.normal(debt * 0.05 * tgt, 500))
            credit_sum  = debt * RNG.uniform(1.0, 1.3)
            bureau_rows.append({
                'SK_ID_CURR':              sk_id,
                'SK_ID_BUREAU':            bur_id,
                'CREDIT_TYPE':             RNG.choice(['Consumer credit', 'Credit card', 'Mortgage']),
                'AMT_CREDIT_SUM':          round(credit_sum, 2),
                'AMT_CREDIT_SUM_DEBT':     round(debt, 2),
                'AMT_CREDIT_SUM_OVERDUE':  round(overdue, 2),
                'DAYS_CREDIT':             -int(RNG.integers(30, 2000)),
                'CREDIT_ACTIVE':           RNG.choice(['Active', 'Closed']),
            })
            bur_id += 1
    bur_raw = pd.DataFrame(bureau_rows)

    # ---- previous_application table (0–5 rows per applicant) ----
    prev_rows = []
    prev_id = 300_000
    for sk_id, tgt in zip(sk_ids, target):
        n = int(RNG.integers(0, 6))   # 0–5 prior applications
        for _ in range(n):
            amt_credit  = RNG.lognormal(10, 0.6)
            amt_annuity = amt_credit / RNG.uniform(18, 60)
            prev_rows.append({
                'SK_ID_CURR':     sk_id,
                'SK_ID_PREV':     prev_id,
                'NAME_CONTRACT_TYPE': RNG.choice(['Cash loans', 'Revolving loans']),
                'AMT_CREDIT':     round(amt_credit, 2),
                'AMT_ANNUITY':    round(amt_annuity, 2),
                'AMT_APPLICATION': round(amt_credit * RNG.uniform(0.8, 1.0), 2),
                'DAYS_DECISION':  -int(RNG.integers(30, 1500)),
                'NAME_CONTRACT_STATUS': RNG.choice(['Approved', 'Refused', 'Canceled']),
            })
            prev_id += 1
    prev_raw = pd.DataFrame(prev_rows)

    # ---- Save to Drive ----
    app_raw.to_csv(os.path.join(DATA_DIR, 'application.csv'), index=False)
    bur_raw.to_csv(os.path.join(DATA_DIR, 'bureau.csv'), index=False)
    prev_raw.to_csv(os.path.join(DATA_DIR, 'previous_application.csv'), index=False)
    open(CKPT_DATA, 'w').close()
    print('Saved to Drive.')

default_rate = app_raw['TARGET'].mean()
print(f'\napplication:          {len(app_raw):>7,} rows   (default rate: {default_rate:.1%})')
print(f'bureau:               {len(bur_raw):>7,} rows   ({len(bur_raw)/len(app_raw):.1f} avg per applicant)')
print(f'previous_application: {len(prev_raw):>7,} rows   ({len(prev_raw)/len(app_raw):.1f} avg per applicant)')

In [ ]:
# ============================================================
# Cell 4 — Traceprop LineageTracker implementation
# ============================================================
import pandas as pd
import numpy as np
import time
from collections import defaultdict
from typing import Dict, List, Optional


def _inner_dict():
    """Named factory so LineageGraph is picklable."""
    return defaultdict(list)


class SourceTable:
    def __init__(self, name: str, df: pd.DataFrame, key_col: Optional[str] = None):
        self.name    = name
        self.df      = df.reset_index(drop=True)
        self.key_col = key_col
        self.n_rows  = len(self.df)

    def __repr__(self):
        return f'SourceTable({self.name!r}, n={self.n_rows:,}, key={self.key_col!r})'


class LineageGraph:
    """
    Tracks, for each training sample i, which source rows in each
    SourceTable contributed to it during ETL.

    Uses named factory functions (not lambdas) so the object is picklable.
    """

    def __init__(self):
        self._lineage: Dict[int, Dict[str, List[int]]] = defaultdict(_inner_dict)
        self._sources: Dict[str, SourceTable] = {}
        self._build_time_s: float = 0.0

    def register(self, source: SourceTable):
        self._sources[source.name] = source

    def record(self, sample_idx: int, table_name: str, row_indices: List[int]):
        self._lineage[sample_idx][table_name].extend(row_indices)

    def ancestors(self, sample_idx: int) -> Dict[str, List[int]]:
        return dict(self._lineage[sample_idx])

    def trace_to_file(self, sample_idx: int, table_name: str) -> pd.DataFrame:
        row_ids = self._lineage[sample_idx].get(table_name, [])
        if not row_ids:
            return pd.DataFrame()
        return self._sources[table_name].df.iloc[row_ids]

    def n_samples(self) -> int:
        return len(self._lineage)

    def source_names(self) -> List[str]:
        return list(self._sources.keys())

    def stats(self) -> dict:
        rows_per_sample = [
            sum(len(v) for v in self._lineage[i].values())
            for i in self._lineage
        ]
        return {
            'n_samples': self.n_samples(),
            'n_source_tables': len(self._sources),
            'source_tables': self.source_names(),
            'avg_source_rows_per_sample': float(np.mean(rows_per_sample)) if rows_per_sample else 0.0,
            'max_source_rows_per_sample': int(np.max(rows_per_sample)) if rows_per_sample else 0,
        }


print('LineageTracker defined (pickle-safe).')

In [ ]:
# ============================================================
# Cell 4b — Clear stale ETL checkpoint (run once after fixing
#            the pickle error, then skip on future runs)
# ============================================================
import os

stale = os.path.join(CKPT_DIR, 'etl_complete.pkl')
if os.path.exists(stale):
    os.remove(stale)
    print(f'Deleted stale checkpoint: {stale}')
else:
    print('No stale checkpoint found — nothing to delete.')

In [ ]:
# ============================================================
# Cell 5 — ETL pipeline with provenance tracking
# ============================================================
import pickle, time

CKPT_ETL = os.path.join(CKPT_DIR, 'etl_complete.pkl')

if os.path.exists(CKPT_ETL):
    print('ETL checkpoint found — loading from Drive...')
    with open(CKPT_ETL, 'rb') as f:
        ckpt = pickle.load(f)
    X_tp          = ckpt['X_tp']
    y             = ckpt['y']
    lineage       = ckpt['lineage']
    etl_time_tp   = ckpt['etl_time_tp']
    etl_time_base = ckpt['etl_time_base']
    print(f'Loaded: X shape={X_tp.shape}, n_samples={lineage.n_samples():,}')

else:
    print('Running ETL pipeline...')

    FEATURE_COLS = [
        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
        'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_FAM_MEMBERS',
        'bureau_n_credits', 'bureau_debt_sum', 'bureau_overdue_mean',
        'prev_n_apps', 'prev_credit_mean', 'prev_annuity_mean',
    ]

    def run_etl_baseline(app, bur, prev):
        bur_agg = bur.groupby('SK_ID_CURR').agg(
            bureau_n_credits=('SK_ID_BUREAU', 'count'),
            bureau_debt_sum=('AMT_CREDIT_SUM_DEBT', 'sum'),
            bureau_overdue_mean=('AMT_CREDIT_SUM_OVERDUE', 'mean'),
        ).reset_index()
        prev_agg = prev.groupby('SK_ID_CURR').agg(
            prev_n_apps=('SK_ID_PREV', 'count'),
            prev_credit_mean=('AMT_CREDIT', 'mean'),
            prev_annuity_mean=('AMT_ANNUITY', 'mean'),
        ).reset_index()
        df = app.merge(bur_agg, on='SK_ID_CURR', how='left')
        df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
        X = df[FEATURE_COLS].fillna(0).values.astype(np.float32)
        y = df['TARGET'].values
        return X, y, df

    def _build_child_index(child_sk: np.ndarray, child_row: np.ndarray,
                           sk_to_sample: dict) -> dict:
        """
        Build {sample_idx: [child_row_ids]} using numpy sort+split.
        Avoids per-group Python overhead of pandas groupby.
        """
        # Map SK_ID_CURR → sample_idx (vectorised via pandas map)
        sample_idx_arr = np.array(
            [sk_to_sample.get(sk, -1) for sk in child_sk], dtype=np.int32
        )
        valid = sample_idx_arr >= 0
        s_idx = sample_idx_arr[valid]
        r_idx = child_row[valid]

        # Sort by sample_idx
        order      = np.argsort(s_idx, kind='stable')
        s_sorted   = s_idx[order]
        r_sorted   = r_idx[order]

        # Split into per-sample lists using np.split
        unique_s, counts = np.unique(s_sorted, return_counts=True)
        splits = np.split(r_sorted, np.cumsum(counts)[:-1])

        return {int(s): grp.tolist() for s, grp in zip(unique_s, splits)}

    # ---- Baseline ----
    t0 = time.perf_counter()
    X_base, y_base, df_base = run_etl_baseline(
        app_raw.copy(), bur_raw.copy(), prev_raw.copy()
    )
    etl_time_base = time.perf_counter() - t0
    print(f'  Baseline ETL:   {etl_time_base:.3f}s')

    # ---- Traceprop ETL ----
    t0 = time.perf_counter()

    lineage = LineageGraph()
    lineage.register(SourceTable('application',          app_raw,  key_col='SK_ID_CURR'))
    lineage.register(SourceTable('bureau',               bur_raw,  key_col='SK_ID_CURR'))
    lineage.register(SourceTable('previous_application', prev_raw, key_col='SK_ID_CURR'))

    X_tp, y, df_tp = run_etl_baseline(
        app_raw.copy(), bur_raw.copy(), prev_raw.copy()
    )

    # SK_ID_CURR → sample_idx lookup
    sk_to_sample = {sk: i for i, sk in enumerate(df_tp['SK_ID_CURR'].values)}

    # application: 1-to-1 (vectorised)
    app_sk_to_row = {sk: i for i, sk in enumerate(app_raw['SK_ID_CURR'].values)}
    for sample_idx, sk in enumerate(df_tp['SK_ID_CURR'].values):
        lineage._lineage[sample_idx]['application'] = [app_sk_to_row[sk]]

    # bureau: many-to-1 (numpy sort+split)
    bur_child = _build_child_index(
        bur_raw['SK_ID_CURR'].values,
        bur_raw.index.values.astype(np.int32),
        sk_to_sample,
    )
    for s, rows in bur_child.items():
        lineage._lineage[s]['bureau'] = rows

    # previous_application
    prev_child = _build_child_index(
        prev_raw['SK_ID_CURR'].values,
        prev_raw.index.values.astype(np.int32),
        sk_to_sample,
    )
    for s, rows in prev_child.items():
        lineage._lineage[s]['previous_application'] = rows

    etl_time_tp = time.perf_counter() - t0
    lineage._build_time_s = etl_time_tp

    ckpt = {
        'X_tp': X_tp, 'y': y,
        'lineage': lineage,
        'etl_time_tp': etl_time_tp,
        'etl_time_base': etl_time_base,
    }
    with open(CKPT_ETL, 'wb') as f:
        pickle.dump(ckpt, f)
    print(f'  Traceprop ETL:  {etl_time_tp:.3f}s')
    print(f'  Checkpoint saved → {CKPT_ETL}')

overhead_ratio = etl_time_tp / etl_time_base
print(f'\nETL overhead ratio (Traceprop / baseline): {overhead_ratio:.3f}x')
print('Lineage graph stats:')
for k, v in lineage.stats().items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 6 — Train logistic regression
#
# Checkpoint: saves model to Drive.
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

CKPT_MODEL = os.path.join(CKPT_DIR, 'logreg_model.pkl')

if os.path.exists(CKPT_MODEL):
    print('Model checkpoint found — loading from Drive...')
    with open(CKPT_MODEL, 'rb') as f:
        ckpt_m = pickle.load(f)
    clf     = ckpt_m['clf']
    scaler  = ckpt_m['scaler']
    X_train = ckpt_m['X_train']
    X_test  = ckpt_m['X_test']
    y_train = ckpt_m['y_train']
    y_test  = ckpt_m['y_test']
    train_idx = ckpt_m['train_idx']
    test_idx  = ckpt_m['test_idx']
    print(f'Model loaded. Test AUC: {ckpt_m["test_auc"]:.4f}')

else:
    print('Training logistic regression...')

    # Train/test split — preserve original indices for lineage lookup
    all_idx = np.arange(len(X_tp))
    train_idx, test_idx = train_test_split(all_idx, test_size=0.2,
                                           random_state=42, stratify=y)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_tp[train_idx])
    X_test  = scaler.transform(X_tp[test_idx])
    y_train = y[train_idx]
    y_test  = y[test_idx]

    clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                              class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)

    probs     = clf.predict_proba(X_test)[:, 1]
    test_auc  = roc_auc_score(y_test, probs)
    print(f'Test AUC: {test_auc:.4f}')

    ckpt_m = {
        'clf': clf, 'scaler': scaler,
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'train_idx': train_idx, 'test_idx': test_idx,
        'test_auc': test_auc,
    }
    with open(CKPT_MODEL, 'wb') as f:
        pickle.dump(ckpt_m, f)
    print(f'Model checkpoint saved → {CKPT_MODEL}')

print(f'\nTraining set: {len(X_train):,}  Test set: {len(X_test):,}')

In [ ]:
# ============================================================
# Cell 7 — Traceprop-LL: per-sample last-layer gradients
#
# For logistic regression, the last-layer gradient equals the
# full gradient (exact, not approximate).
#
# grad_i = (p_i - y_i) * x_i   where p_i = sigmoid(W x_i)
# Attribution score: s_i = dot(grad_i, grad_test) / norm
#
# Checkpoint: saves gradient store to Drive.
# ============================================================

CKPT_GRADS = os.path.join(CKPT_DIR, 'gradient_store.pkl')

if os.path.exists(CKPT_GRADS):
    print('Gradient store checkpoint found — loading from Drive...')
    with open(CKPT_GRADS, 'rb') as f:
        ckpt_g = pickle.load(f)
    G        = ckpt_g['G']
    grad_time = ckpt_g['grad_time']
    print(f'G shape: {G.shape}  (computed in {grad_time:.3f}s)')

else:
    print('Computing per-sample last-layer gradients (Traceprop-LL)...')

    W = clf.coef_[0]          # (n_features,)
    b = clf.intercept_[0]     # scalar

    t0 = time.perf_counter()

    # Predicted probabilities on training set
    logits = X_train @ W + b              # (n_train,)
    p      = 1.0 / (1.0 + np.exp(-logits))  # sigmoid

    # Per-sample gradient: (p_i - y_i) * x_i  → shape (n_train, n_features)
    residuals = (p - y_train).astype(np.float32)  # (n_train,)
    G = residuals[:, None] * X_train              # (n_train, n_features)

    grad_time = time.perf_counter() - t0
    print(f'Gradient computation: {grad_time:.3f}s')
    print(f'Gradient store shape: {G.shape}  '
          f'({G.nbytes / 1e6:.1f} MB)')

    ckpt_g = {'G': G, 'grad_time': grad_time}
    with open(CKPT_GRADS, 'wb') as f:
        pickle.dump(ckpt_g, f)
    print(f'Checkpoint saved → {CKPT_GRADS}')

In [ ]:
# ============================================================
# Cell 8 — Compute attribution scores for test samples
#
# s_i = dot(g_i, g_test) / max_j |dot(g_j, g_test)|
# ============================================================

def compute_attribution_scores(G_train, X_test_sample, W, b, y_test_sample):
    """Return L-inf normalised attribution scores for one test point."""
    logit_test = X_test_sample @ W + b
    p_test     = 1.0 / (1.0 + np.exp(-logit_test))
    g_test     = (p_test - y_test_sample) * X_test_sample  # (n_features,)

    raw_scores = G_train @ g_test           # (n_train,)
    max_abs    = np.max(np.abs(raw_scores))
    if max_abs > 0:
        scores = raw_scores / max_abs
    else:
        scores = raw_scores
    return scores, g_test


W = clf.coef_[0]
b = clf.intercept_[0]

# Demo: compute for first 5 test samples
DEMO_SAMPLES = 5
all_scores = []
for i in range(DEMO_SAMPLES):
    scores, _ = compute_attribution_scores(
        G, X_test[i], W, b, y_test[i]
    )
    all_scores.append(scores)

print(f'Attribution scores computed for {DEMO_SAMPLES} test samples.')
print(f'Scores shape: {all_scores[0].shape}')
print(f'Score range: [{all_scores[0].min():.4f}, {all_scores[0].max():.4f}]')

In [ ]:
# ============================================================
# Cell 9 — trace_to_file(): end-to-end provenance query
#
# Given test sample t:
#   1. Compute attribution scores → top-k influential training samples
#   2. For each top training sample, call lineage.trace_to_file()
#   3. Return which source rows in each table were responsible
#
# This is the core VLDB demo: a single query that crosses
# 3 source tables to explain one prediction.
# ============================================================

def trace_to_file(test_sample_idx: int, top_k: int = 5):
    """
    Full provenance trace for test sample test_sample_idx.

    Returns:
      report: dict with top-k training samples and their source rows.
    """
    scores, _ = compute_attribution_scores(
        G, X_test[test_sample_idx], W, b, y_test[test_sample_idx]
    )

    # Top-k most influential training samples (by absolute score)
    top_train_idx = np.argsort(np.abs(scores))[::-1][:top_k]

    # For each top training sample, resolve back to original dataset index
    # (train_idx maps position-in-X_train → position-in-full-dataset)
    report = {
        'test_sample_idx': test_sample_idx,
        'predicted_class': int(clf.predict(X_test[test_sample_idx:test_sample_idx+1])[0]),
        'predicted_prob':  float(clf.predict_proba(X_test[test_sample_idx:test_sample_idx+1])[0, 1]),
        'true_label':      int(y_test[test_sample_idx]),
        'top_k_training_samples': [],
    }

    for rank, pos_in_train in enumerate(top_train_idx):
        dataset_idx = train_idx[pos_in_train]   # index into full X_tp / lineage
        score       = float(scores[pos_in_train])
        ancestors   = lineage.ancestors(dataset_idx)

        entry = {
            'rank':        rank + 1,
            'dataset_idx': int(dataset_idx),
            'score':       score,
            'source_rows': {},
        }
        for table_name, row_ids in ancestors.items():
            entry['source_rows'][table_name] = {
                'n_rows': len(row_ids),
                'row_ids': row_ids[:3],  # show first 3
            }
        report['top_k_training_samples'].append(entry)

    return report


# Demo query: trace prediction for test sample 0
t0 = time.perf_counter()
report = trace_to_file(test_sample_idx=0, top_k=5)
query_time_ms = (time.perf_counter() - t0) * 1000

print('=== trace_to_file() demo ===')
print(f'Test sample 0:')
print(f'  Predicted class:  {report["predicted_class"]}')
print(f'  Predicted prob:   {report["predicted_prob"]:.4f}')
print(f'  True label:       {report["true_label"]}')
print(f'  Query time:       {query_time_ms:.3f} ms')
print()
print('Top-3 most influential training samples:')
for e in report['top_k_training_samples'][:3]:
    print(f'  Rank {e["rank"]:d}: dataset_idx={e["dataset_idx"]:6d}  score={e["score"]:+.4f}')
    for tbl, info in e['source_rows'].items():
        print(f'           {tbl:30s}  {info["n_rows"]:3d} row(s)  row_ids={info["row_ids"]}')

In [ ]:
# ============================================================
# Cell 10 — Query latency benchmark
#
# Measures wall-clock time for:
#   (a) ancestors(sample_idx) — lineage DAG traversal
#   (b) trace_to_file(test_idx) — full end-to-end provenance query
#
# Across N_REPS random samples to get stable estimates.
# ============================================================

N_REPS = 200
rng = np.random.default_rng(0)

# (a) ancestors() latency
sample_ids = rng.integers(0, len(X_tp), size=N_REPS)
t0 = time.perf_counter()
for idx in sample_ids:
    _ = lineage.ancestors(int(idx))
ancestors_ms = (time.perf_counter() - t0) / N_REPS * 1000

# (b) trace_to_file() latency (includes attribution + lineage)
test_ids = rng.integers(0, len(X_test), size=N_REPS)
t0 = time.perf_counter()
for idx in test_ids:
    _ = trace_to_file(int(idx), top_k=5)
trace_ms = (time.perf_counter() - t0) / N_REPS * 1000

print('=== Query Latency ===')
print(f'  ancestors(sample_idx)        : {ancestors_ms:.4f} ms  (avg over {N_REPS} calls)')
print(f'  trace_to_file(test_idx, k=5) : {trace_ms:.4f} ms  (avg over {N_REPS} calls)')
print()
print('Lineage graph stats:')
stats = lineage.stats()
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 11 — Source table contribution analysis
#
# Which source tables contribute the most influential
# training samples for a given prediction?
# Computes per-table mean attribution score.
#
# This produces the provenance breakdown figure for VLDB.
# ============================================================

def source_table_attribution(test_sample_idx: int, top_k: int = 50):
    """
    For a given test prediction, compute the mean absolute attribution
    score grouped by source table.

    Returns a dict: {table_name: mean_abs_attribution}
    """
    scores, _ = compute_attribution_scores(
        G, X_test[test_sample_idx], W, b, y_test[test_sample_idx]
    )
    top_idx = np.argsort(np.abs(scores))[::-1][:top_k]

    table_scores = defaultdict(list)
    for pos in top_idx:
        dataset_idx = train_idx[pos]
        abs_score   = abs(float(scores[pos]))
        ancestors   = lineage.ancestors(int(dataset_idx))
        for tbl in ancestors:
            if ancestors[tbl]:   # table contributed rows
                table_scores[tbl].append(abs_score)

    return {tbl: float(np.mean(v)) for tbl, v in table_scores.items()}


# Aggregate over first 100 test samples
N_DEMO = 100
all_table_scores = defaultdict(list)
for i in range(N_DEMO):
    contrib = source_table_attribution(i, top_k=20)
    for tbl, score in contrib.items():
        all_table_scores[tbl].append(score)

print('=== Source Table Attribution (mean over 100 test samples) ===')
for tbl, scores_list in sorted(all_table_scores.items(),
                                key=lambda x: -np.mean(x[1])):
    print(f'  {tbl:35s}  mean_abs_score={np.mean(scores_list):.4f}  '
          f'n={len(scores_list)}')

In [ ]:
# ============================================================
# Cell 12 — Compile results + save JSON
#
# Saves canonical results for the VLDB paper (Section 6.6)
# to both Drive and local results/ directory.
# ============================================================
import json

results = {
    'experiment': 'exp5_multisource_credit_provenance',
    'dataset': 'Synthetic credit risk (Home Credit schema, 3 tables)',
    'n_source_tables': 3,
    'source_tables': ['application', 'bureau', 'previous_application'],
    'n_apps':    int(len(X_tp)),
    'n_train':   int(len(X_train)),
    'n_test':    int(len(X_test)),
    'n_features': int(X_tp.shape[1]),

    'etl_overhead': {
        'baseline_s':     round(etl_time_base, 4),
        'traceprop_s':    round(etl_time_tp, 4),
        'overhead_ratio': round(overhead_ratio, 4),
    },

    'lineage_graph': lineage.stats(),

    'query_latency_ms': {
        'ancestors':     round(ancestors_ms, 4),
        'trace_to_file': round(trace_ms, 4),
    },

    'gradient_store': {
        'method':         'Traceprop-LL (last-layer, exact for logistic regression)',
        'compute_time_s': round(grad_time, 4),
        'shape':          list(G.shape),
        'size_mb':        round(G.nbytes / 1e6, 2),
    },

    'source_table_attribution': {
        tbl: round(float(np.mean(v)), 4)
        for tbl, v in all_table_scores.items()
    },

    'demo_trace': {
        'test_sample_idx': 0,
        'predicted_prob':  round(report['predicted_prob'], 4),
        'predicted_class': report['predicted_class'],
        'true_label':      report['true_label'],
        'top_1_source_rows': {
            tbl: info['n_rows']
            for tbl, info in report['top_k_training_samples'][0]['source_rows'].items()
        },
    },
}

# Save to Drive
drive_path = os.path.join(RESULTS_DIR, 'exp5_homecredit_provenance.json')
with open(drive_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to Drive → {drive_path}')

# Save locally for git commit
local_path = 'results/exp5_homecredit_provenance.json'
os.makedirs('results', exist_ok=True)
with open(local_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved locally → {local_path}')

print()
print('=== FINAL RESULTS (copy into VLDB Section 6.6) ===')
print(f'  Source tables:                   {results["n_source_tables"]}')
print(f'  Applicants (training samples):   {results["n_train"]:,}')
print(f'  ETL overhead ratio:              {overhead_ratio:.3f}x')
print(f'  Lineage build time:              {etl_time_tp:.3f}s')
print(f'  ancestors() latency:             {ancestors_ms:.4f} ms')
print(f'  trace_to_file() latency:         {trace_ms:.4f} ms')
print(f'  Avg source rows / sample:        {lineage.stats()["avg_source_rows_per_sample"]:.1f}')
print(f'  Gradient store:                  {G.nbytes/1e6:.1f} MB  ({grad_time:.3f}s)')

## Results for VLDB Section 6.6

After running all cells, copy the numbers from Cell 12 output into the paper.

**Table to add to paper:**
```
Metric                          Value
─────────────────────────────────────
Source tables                   3
Training samples                ~16,000
ETL overhead ratio              X.XXx
Lineage build time              X.Xs
ancestors() latency             X.XX ms
trace_to_file() latency         X.XX ms
Avg source rows / sample        XX.X
```

**Narrative claim to add:**
> A single `trace_to_file()` query resolves in <X ms, crossing 3 source tables
> (application, bureau, previous_application) to return the originating rows
> that most influenced a credit risk prediction.

**Checkpoints on Drive:**
- `data_downloaded.flag` — raw CSVs downloaded
- `etl_complete.pkl` — processed features + lineage graph
- `logreg_model.pkl` — trained classifier
- `gradient_store.pkl` — per-sample gradients
